In [ ]:
import re
import os
import sys
import math
import numpy as np
import das4whales as dw
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1 import make_axes_locatable
from datetime import datetime

DAS_APL_NAS_SERVER_PATH = "Z:/DAS-APL"
sys.path.append(DAS_APL_NAS_SERVER_PATH)

import processing.spectral_decomposition
import processing.normalize_bands
import processing.crop

In [ ]:
FILES = [
    'Z:/DAS-APL/data/OOI_RCA2021/Optasence/NorthCable/North-C1-LR-P1kHz-GL50m-Sp2m-FS200Hz_2021-11-04T015502Z.h5',
    'Z:/DAS-APL/data/OOI_RCA2021/Optasence/NorthCable/North-C1-LR-P1kHz-GL50m-Sp2m-FS200Hz_2021-11-04T015602Z.h5',
    'Z:/DAS-APL/data/OOI_RCA2021/Optasence/NorthCable/North-C1-LR-P1kHz-GL50m-Sp2m-FS200Hz_2021-11-04T015702Z.h5',
    'Z:/DAS-APL/data/OOI_RCA2021/Optasence/NorthCable/North-C1-LR-P1kHz-GL50m-Sp2m-FS200Hz_2021-11-04T015802Z.h5',
    'Z:/DAS-APL/data/OOI_RCA2021/Optasence/NorthCable/North-C1-LR-P1kHz-GL50m-Sp2m-FS200Hz_2021-11-04T015902Z.h5',
    'Z:/DAS-APL/data/OOI_RCA2021/Optasence/NorthCable/North-C1-LR-P1kHz-GL50m-Sp2m-FS200Hz_2021-11-04T020002Z.h5',
    'Z:/DAS-APL/data/OOI_RCA2021/Optasence/NorthCable/North-C1-LR-P1kHz-GL50m-Sp2m-FS200Hz_2021-11-04T020102Z.h5',
    'Z:/DAS-APL/data/OOI_RCA2021/Optasence/NorthCable/North-C1-LR-P1kHz-GL50m-Sp2m-FS200Hz_2021-11-04T020202Z.h5',
    'Z:/DAS-APL/data/OOI_RCA2021/Optasence/NorthCable/North-C1-LR-P1kHz-GL50m-Sp2m-FS200Hz_2021-11-04T022402Z.h5'
]

# Frequency bands [Hz]
B1 = [3, 15]    # Low frequency noise
B2 = [16, 20]   # LF
B3 = [20, 28]   # HF

# Crop window
T_MIN, T_MAX = 0, 60          # seconds
D_MIN, D_MAX = 20e3, 60e3     # meters

# Normalisation percentile
PERC = 90

# Layout: number of columns (rows computed automatically)
N_COLS = 3

# Font sizes
LABEL_SIZE  = 15
TICK_SIZE   = 14
TITLE_SIZE  = 14
LEGEND_SIZE = 12

# Output paths (set to None to skip)
OUT_PDF = "Z:/DAS-APL/exp/multispectral_representation/Exp01-X2.pdf"
OUT_PNG = "Z:/DAS-APL/exp/multispectral_representation/Exp01-X2.png"

In [ ]:
def process_file(filepath):
    """Load a DAS file and return the cropped RGB array + axes vectors."""
    metadata = dw.data_handle.get_acquisition_parameters(filepath, interrogator='optasense')
    fs = metadata['fs']
    dx = metadata['dx']
    nx = metadata['nx']

    selected_channels_m = [0, nx * dx, dx]
    selected_channels   = [int(x // dx) for x in selected_channels_m]

    tr, time, dist, _ = dw.data_handle.load_das_data(filepath, selected_channels, metadata)

    tr_b01, tr_b02, tr_b03 = spectral_decomposition(tr, fs, [B1, B2, B3])

    # Crop
    tr_b01_c, dist_c, time_c = crop(tr_b01, dist, time, D_MIN, D_MAX, T_MIN, T_MAX)
    tr_b02_c, _,      _      = crop(tr_b02, dist, time, D_MIN, D_MAX, T_MIN, T_MAX)
    tr_b03_c, _,      _      = crop(tr_b03, dist, time, D_MIN, D_MAX, T_MIN, T_MAX)

    # Normalise & stack
    r   = normalize_band(tr_b03_c, perc=PERC)
    g   = normalize_band(tr_b02_c, perc=PERC)
    b   = normalize_band(tr_b01_c, perc=PERC)
    rgb = np.stack([r, g, b], axis=-1)

    return rgb, time_c, dist_c


# def short_title(filepath):
#     return os.path.splitext(os.path.basename(filepath))[0]

def short_title(filepath):
    name = os.path.splitext(os.path.basename(filepath))[0]  # sin extensión
    match = re.search(r'(\d{4}-\d{2}-\d{2}T\d{6}Z)', name)
    if match:
        dt = datetime.strptime(match.group(1), '%Y-%m-%dT%H%M%SZ')
        return dt.strftime('%Y-%m-%d  %H:%M:%S UTC')
    return name

In [ ]:
n_files = len(FILES)
n_cols  = min(N_COLS, n_files)
n_rows  = math.ceil(n_files / n_cols)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(6 * n_cols, 5.5 * n_rows),
    squeeze=False
)

legend_handles = [
    Patch(color='red',   label=f'Red:   {B3[0]}–{B3[1]} Hz'),
    Patch(color='green', label=f'Green: {B2[0]}–{B2[1]} Hz'),
    Patch(color='blue',  label=f'Blue:  {B1[0]}–{B1[1]} Hz'),
]

for idx, filepath in enumerate(FILES):
    row = idx // n_cols
    col = idx % n_cols
    ax  = axes[row][col]

    print(f"[{idx+1}/{n_files}] Processing: {os.path.basename(filepath)}")

    rgb, time_c, dist_c = process_file(filepath)

    ax.imshow(
        rgb,
        aspect='auto',
        origin='lower',
        extent=[time_c[0], time_c[-1],
                dist_c[0] / 1e3, dist_c[-1] / 1e3]
    )

    ax.set_title(short_title(filepath), fontsize=TITLE_SIZE, pad=6)
    ax.tick_params(axis='both', which='major', labelsize=TICK_SIZE)

    # Only left-column axes get a y-label
    if col == 0:
        ax.set_ylabel('Distance [km]', fontsize=LABEL_SIZE)

    # Only bottom-row axes get an x-label
    if row == n_rows - 1:
        ax.set_xlabel('Time [s]', fontsize=LABEL_SIZE)

    # Legend on the last populated subplot only
    if idx == n_files - 1:
        ax.legend(handles=legend_handles, loc='lower right',
                  framealpha=0.7, fontsize=LEGEND_SIZE)

for empty in range(n_files, n_rows * n_cols):
    axes[empty // n_cols][empty % n_cols].set_visible(False)

plt.tight_layout()

if OUT_PDF:
    fig.savefig(OUT_PDF, format="pdf", dpi=300, bbox_inches="tight", pad_inches=0.1)
    print(f"Saved PDF: {OUT_PDF}")

if OUT_PNG:
    fig.savefig(OUT_PNG, format="png", dpi=300, bbox_inches="tight", pad_inches=0.1)
    print(f"Saved PNG: {OUT_PNG}")

plt.show()
